# LLM Benchmark — GPT vs Claude vs Gemini on Databricks

This notebook sends identical prompts to all three providers via the **Databricks AI Gateway** and compares:
- Token usage (each model's own tokenizer)
- Vendor AI cost (no DBU charges)
- Response time
- Accuracy score


## 1 · Setup — credentials & install

In [ ]:
# Install dependencies if running outside the project venv
# %pip install openai pandas matplotlib -q

In [ ]:
import os
import sys

# ── Add project root to path so we can import src.llm_benchmark ──
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd()))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# ── Databricks credentials ────────────────────────────────────────
# Set these before running, or export them in your shell before launching Jupyter.
os.environ.setdefault("DATABRICKS_HOST",  "https://adb-<workspace-id>.azuredatabricks.net")
os.environ.setdefault("DATABRICKS_TOKEN", "dapi<your-personal-access-token>")

print("Host :", os.environ["DATABRICKS_HOST"])
print("Token:", os.environ["DATABRICKS_TOKEN"][:8] + "…")

## 2 · Models & pricing

In [ ]:
import pandas as pd
from src.llm_benchmark.config import MODELS

df_models = pd.DataFrame(MODELS)[[
    "display_name", "provider", "input_price_per_1m", "output_price_per_1m"
]].rename(columns={
    "display_name":        "Model",
    "provider":            "Provider",
    "input_price_per_1m":  "Input $/1M tokens",
    "output_price_per_1m": "Output $/1M tokens",
})

df_models.style.format({
    "Input $/1M tokens":  "${:.3f}",
    "Output $/1M tokens": "${:.3f}",
}).set_caption("Vendor AI token prices (no DBU costs)")

## 3 · Test prompts

In [ ]:
from src.llm_benchmark.prompts import TEST_PROMPTS

df_prompts = pd.DataFrame(TEST_PROMPTS)[["id", "category", "prompt"]].rename(columns={
    "id":       "Prompt ID",
    "category": "Category",
    "prompt":   "Question",
})

df_prompts.style.set_properties(**{"text-align": "left"}).hide(axis="index")

## 4 · Run benchmark

> **Tip:** change `selected_providers` to test only a subset, e.g. `["openai"]`.

In [ ]:
from src.llm_benchmark.benchmark import run_benchmark

# Filter which providers to include
selected_providers = ["openai", "anthropic", "google"]   # change as needed
models_to_run = [m for m in MODELS if m["provider"] in selected_providers]

print(f"Running {len(models_to_run)} models × {len(TEST_PROMPTS)} prompts "
      f"= {len(models_to_run) * len(TEST_PROMPTS)} API calls …\n")

results = run_benchmark(models=models_to_run, max_tokens=256)

print(f"\nDone — {len(results)} results collected.")

## 5 · Results — detail table

In [ ]:
from dataclasses import asdict

df = pd.DataFrame([asdict(r) for r in results])

detail_cols = [
    "model_display_name", "prompt_id", "prompt_category",
    "input_tokens", "output_tokens", "total_tokens",
    "ai_cost_usd", "response_time_sec", "accuracy_label", "accuracy_score",
]

df_detail = df[detail_cols].rename(columns={
    "model_display_name": "Model",
    "prompt_id":          "Prompt",
    "prompt_category":    "Category",
    "input_tokens":       "In Tokens",
    "output_tokens":      "Out Tokens",
    "total_tokens":       "Total Tokens",
    "ai_cost_usd":        "AI Cost ($)",
    "response_time_sec":  "Time (s)",
    "accuracy_label":     "Accuracy",
    "accuracy_score":     "Score",
})

ACCURACY_COLORS = {
    "Excellent": "background-color: #c6efce",
    "Good":      "background-color: #ffeb9c",
    "Partial":   "background-color: #ffcc99",
    "Poor":      "background-color: #ffc7ce",
}

def color_accuracy(val):
    return ACCURACY_COLORS.get(val, "")

df_detail.style \
    .format({"AI Cost ($)": "${:.6f}", "Time (s)": "{:.2f}s", "Score": "{:.2f}"}) \
    .applymap(color_accuracy, subset=["Accuracy"]) \
    .set_caption("Detail: every model × every prompt") \
    .hide(axis="index")

## 6 · Summary — aggregated per model

In [ ]:
summary = (
    df.groupby("model_display_name")
    .agg(
        Prompts        = ("prompt_id",         "count"),
        Errors         = ("error",              lambda x: x.notna().sum()),
        Avg_Tokens     = ("total_tokens",       "mean"),
        Total_AI_Cost  = ("ai_cost_usd",        "sum"),
        Avg_Time_sec   = ("response_time_sec",  "mean"),
        Avg_Accuracy   = ("accuracy_score",     "mean"),
    )
    .reset_index()
    .rename(columns={"model_display_name": "Model"})
    .sort_values("Avg_Accuracy", ascending=False)
    .reset_index(drop=True)
)

summary.style \
    .format({
        "Avg_Tokens":    "{:.0f}",
        "Total_AI_Cost": "${:.6f}",
        "Avg_Time_sec":  "{:.2f}s",
        "Avg_Accuracy":  "{:.2f}",
    }) \
    .background_gradient(subset=["Avg_Accuracy"], cmap="RdYlGn") \
    .background_gradient(subset=["Total_AI_Cost"], cmap="RdYlGn_r") \
    .background_gradient(subset=["Avg_Time_sec"], cmap="RdYlGn_r") \
    .set_caption("Summary: aggregated per model (sorted by accuracy)") \
    .hide(axis="index")

## 7 · Charts

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("LLM Benchmark — Databricks AI Gateway", fontsize=14, fontweight="bold")

models  = summary["Model"]
x       = range(len(models))
colors  = ["#4472C4", "#ED7D31", "#A9D18E", "#FFC000", "#5B9BD5", "#FF0000"]

# ── Accuracy ──
axes[0].bar(x, summary["Avg_Accuracy"], color=colors[:len(models)])
axes[0].set_title("Avg Accuracy Score")
axes[0].set_xticks(x); axes[0].set_xticklabels(models, rotation=30, ha="right")
axes[0].set_ylim(0, 1)
axes[0].set_ylabel("Score (0–1)")
for i, v in enumerate(summary["Avg_Accuracy"]):
    axes[0].text(i, v + 0.02, f"{v:.2f}", ha="center", fontsize=9)

# ── Total AI Cost ──
axes[1].bar(x, summary["Total_AI_Cost"], color=colors[:len(models)])
axes[1].set_title("Total AI Cost (USD)")
axes[1].set_xticks(x); axes[1].set_xticklabels(models, rotation=30, ha="right")
axes[1].set_ylabel("USD")
for i, v in enumerate(summary["Total_AI_Cost"]):
    axes[1].text(i, v + 0.000001, f"${v:.5f}", ha="center", fontsize=8)

# ── Avg Response Time ──
axes[2].bar(x, summary["Avg_Time_sec"], color=colors[:len(models)])
axes[2].set_title("Avg Response Time (s)")
axes[2].set_xticks(x); axes[2].set_xticklabels(models, rotation=30, ha="right")
axes[2].set_ylabel("Seconds")
for i, v in enumerate(summary["Avg_Time_sec"]):
    axes[2].text(i, v + 0.05, f"{v:.2f}s", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig("llm_benchmark_charts.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved → llm_benchmark_charts.png")

## 8 · Accuracy breakdown by prompt category

In [ ]:
pivot = df.pivot_table(
    index="prompt_category",
    columns="model_display_name",
    values="accuracy_score",
    aggfunc="mean"
)

pivot.style \
    .format("{:.2f}") \
    .background_gradient(cmap="RdYlGn", axis=None) \
    .set_caption("Accuracy score by category × model")

## 9 · Export to CSV

In [ ]:
csv_path = "llm_benchmark_results.csv"
df[detail_cols].to_csv(csv_path, index=False)
print(f"Results saved → {csv_path}")

## 10 · Model responses (inspect any call)

In [ ]:
# Change these to inspect a specific model + prompt
INSPECT_MODEL  = "GPT-4o"
INSPECT_PROMPT = "sql_join_explain"

row = df[(df["model_display_name"] == INSPECT_MODEL) & (df["prompt_id"] == INSPECT_PROMPT)]

if row.empty:
    print("No match found — check model name and prompt id.")
else:
    r = row.iloc[0]
    print(f"Model   : {r['model_display_name']}")
    print(f"Prompt  : {r['prompt_text']}")
    print(f"Response: {r['response_text']}")
    print(f"Tokens  : {r['input_tokens']} in / {r['output_tokens']} out")
    print(f"Cost    : ${r['ai_cost_usd']:.6f}")
    print(f"Time    : {r['response_time_sec']:.2f}s")
    print(f"Accuracy: {r['accuracy_label']} ({r['accuracy_score']:.2f})")